## Note on Differencing (Stationarity)
**Question:** Should we manually difference the data (e.g., $y_t - y_{t-1}$) before fitting?

**Answer:** No. Time differencing is handled automatically by the ARIMA model parameters ($d$ for trend, $D$ for seasonality). 
1. `auto_arima` determines the optimal $d$ and $D$ values.
2. `SARIMAX` applies this differencing internally during fitting.
3. `SARIMAX` automatically "integrates" (reverses the differencing) during prediction, so the output forecasts are on the original scale.

Manual differencing would require complex post-processing to reconstruct the scale, especially in a recursive loop. We rely on the model's built-in handling.

# AutoARIMA Test Pipeline
This notebook adapts the LSTM pipeline for AutoARIMA, using the same recursive inference and validation logic.

In [1]:
# Import Required Libraries
import sys
import os
sys.path.append('..') # Add parent directory to path to import model_utils
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Custom modules
from autoarima2 import AutoARIMAPipeline
from model_utils.plots import plot_results
from model_utils.utils import compute_metrics, generate_exogenous_features

In [2]:
DATA_PATH = '../dataset/data_andre.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df = pd.read_feather(DATA_PATH)

#NUM_ITEMS = 100
#df = df[df['item_id'].isin(df['item_id'].unique()[:NUM_ITEMS])]

Loading data from ../dataset/data_andre.feather...


In [3]:
# Filter for specific products if needed
target_products = [26008, 907969,907967]
#target_products = [101054,101125, 101126, 102689, 103633, 103672, 103737, 103776, 103781,103782]
if target_products:
    products = df[df['item_id'].isin(target_products)][['item_id', 'store_id']].drop_duplicates().values
else:
    # Get all unique products from the subset dataset
    products = df[['item_id', 'store_id']].drop_duplicates().values

In [4]:
# -----------------------------------------------------------------------------
# DATA PREPROCESSING
# -----------------------------------------------------------------------------
DATE_COL = 'date'
TARGET_COL = 'value'

# Flatten index if date is trapped in it
if DATE_COL in df.index.names:
    df = df.reset_index()

# Ensure we have proper datetimes
df[DATE_COL] = pd.to_datetime(df[DATE_COL])

# CRITICAL FIX: Sort by Store -> Item -> Date.
# Time-series models absolutely require chronologically sorted data *per series*.
# The previous global sort could mix items up at the boundaries.
df = df.sort_values(['store_id', 'item_id', DATE_COL]).reset_index(drop=True)

# -----------------------------------------------------------------------------
# DATA SPLITTING
# -----------------------------------------------------------------------------
train_size = 455
val_size = 154
forecast_horizon = 152

# -----------------------------------------------------------------------------
# EXOGENOUS FEATURES BUILDER
# -----------------------------------------------------------------------------
EXOG_COLS = [
    # Base calendar
    "day_of_week", "day_of_month", "week_of_year", "week_of_month",
    "month", "quarter", "is_weekend",
    "is_month_start", "is_month_end", "is_quarter_start", "is_quarter_end",
    "is_monday", "is_friday",
    
    # Holidays
    "is_holiday", "is_thanksgiving", "is_black_friday",
    "is_christmas", "is_christmas_eve", "is_new_year_eve",
    "is_pre_holiday_1", "is_pre_holiday_2", "is_pre_holiday_3", "is_pre_holiday_7",
    "is_post_holiday_1", "is_post_holiday_2", "is_post_holiday_3", "is_post_holiday_7",
    "is_bridge_day"
]

# Discover any promotional features natively loaded from the dataset automatically
#promo_cols = [c for c in df.columns if c.startswith("promo_")]
#EXOG_COLS.extend(promo_cols)
#print(f"Dynamically added promotional features: {promo_cols}")

# Generate features
df = generate_exogenous_features(df, date_col=DATE_COL, exog_cols=EXOG_COLS)

print(f"Dataset preprocessed. Total Rows: {len(df)}")

Dataset preprocessed. Total Rows: 1082371


# Grid Search Loop for Products and Seeds
This cell runs AutoARIMA for each product and seed, saving results and plots.

In [5]:
import os
from autoarima import AutoARIMAPipeline
from model_utils.plots import plot_results

pipeline = AutoARIMAPipeline(target_col=TARGET_COL, date_col=DATE_COL)

os.makedirs('grid_search_plots', exist_ok=True)
results = []

for item_id, store_id in products:
    print(f"\nProcessing Item {item_id} Store {store_id}...")
    df_product = df[(df['item_id'] == item_id) & (df['store_id'] == store_id)].copy()
    if DATE_COL in df_product.columns:
        df_product[DATE_COL] = pd.to_datetime(df_product[DATE_COL])
        df_product = df_product.sort_values(DATE_COL).reset_index(drop=True)
    else:
        df_product = df_product.reset_index(drop=True)

    # --- Automatic Seasonality Detection ---
    seasonal_candidates = pipeline.get_seasonal_candidates(
        df=df,
        item_id=item_id,
        store_id=store_id,
        max_lag=1 # Checking up to monthly
    )
    
    print(f"  Selected Seasonality Candidates (m): {seasonal_candidates}")

    # Iterate over EACH candidate to test them individually and generate separate plots
    for m_val in seasonal_candidates:
        print(f"  >> Testing candidate m={m_val}")
        
        # --- Run Approach 1: Lookback (Recursive with sliding window) ---
        print("  Running Lookback Approach...")
        rmse_lb, mae_lb, order_lb, seasonal_order_lb, forecast_lb, y_train, y_test = pipeline.fit_forecast_lookback(
            df=df,
            item_id=item_id,
            store_id=store_id,
            train_size=train_size,
            val_size=val_size,
            forecast_window=forecast_horizon,
            lookback_window=30,  
            seasonal=True if m_val > 1 else False, 
            m=m_val,
            exog_cols=EXOG_COLS,
            maxiter=500  # Increased maxiter to help convergence 
        )
        
        # --- Run Approach 2: Full History (Standard Recursive) ---
        print("  Running Full History Approach...")
        rmse_full, mae_full, order_full, seasonal_order_full, forecast_full = pipeline.fit_forecast_full(
            df=df,
            item_id=item_id,
            store_id=store_id,
            train_size=train_size,
            val_size=val_size,
            forecast_window=forecast_horizon,
            seasonal=True if m_val > 1 else False, 
            m=m_val,
            exog_cols=EXOG_COLS,
            maxiter=500  # Increased maxiter to help convergence
        )

        total_train_val = train_size + val_size
        train_slice = slice(-(total_train_val + forecast_horizon), -forecast_horizon)
        test_slice = slice(-forecast_horizon, None)
        
        dates_train = df_product['date'].iloc[train_slice] if 'date' in df_product.columns else range(len(y_train))
        dates_test = df_product['date'].iloc[test_slice] if 'date' in df_product.columns else range(len(y_train), len(y_train)+len(y_test))

        dates_val = pd.Series([], dtype='datetime64[ns]')
        
        # --- Compare Plot for lookback m ---
        plot_filename_lb = f'grid_search_plots/comparison_item{item_id}_store{store_id}_m{m_val}_lookback.png'
        plot_results(
            train=y_train, val=np.array([]), test=y_test, forecast=forecast_lb, 
            train_index=dates_train, val_index=dates_val, test_index=dates_test, 
            train_losses=[], val_losses=[], metric=f"Lookback Model (m={m_val})",
            title=f'AutoARIMA Lookback Forecast (Item={item_id})', save_path=plot_filename_lb,
            rmse=rmse_lb, mae=mae_lb, bias=0, score=0, df_full=df_product
        )

        
        # --- Compare Plot for full m ---
        plot_filename_full = f'grid_search_plots/comparison_item{item_id}_store{store_id}_m{m_val}_full.png'
        plot_results(
            train=y_train, val=np.array([]), test=y_test, forecast=forecast_full, 
            train_index=dates_train, val_index=dates_val, test_index=dates_test, 
            train_losses=[], val_losses=[], metric=f"Full History Model (m={m_val})",
            title=f'AutoARIMA FullHist Forecast (Item={item_id})', save_path=plot_filename_full,
            rmse=rmse_full, mae=mae_full, bias=0, score=0, df_full=df_product
        )

        print(f"    Saved plots to {plot_filename_lb} and {plot_filename_full}")
        
        results.append({
            'item_id': item_id,
            'store_id': store_id,
            'tested_m': m_val,
            'lookback_rmse': rmse_lb,
            'lookback_mae': mae_lb,
            'lookback_order': order_lb,
            'lookback_seasonal_order': seasonal_order_lb,
            'full_rmse': rmse_full,
            'full_mae': mae_full,
            'full_order': order_full,
            'full_seasonal_order': seasonal_order_full,
            'lookback_plot_path': plot_filename_lb,
            'full_plot_path': plot_filename_full
        })

results_df = pd.DataFrame(results)
results_df.to_csv('autoarima_comparison_results.csv', index=False)
print("Calibration complete. Results saved to 'autoarima_comparison_results.csv'.")


Processing Item 26008 Store 6269...
  Selected Seasonality Candidates (m): [1]
  >> Testing candidate m=1
  Running Lookback Approach...
Lookback Selected model: Order=(2, 0, 2), Seasonal=(0, 0, 0, 0), Trend=n (AIC=4385.50)


c:\Users\Andre Silva\Desktop\Dissertation25-26\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  Running Full History Approach...
Full Selected model: Order=(1, 0, 2), Seasonal=(0, 0, 0, 0), Trend=n (AIC=5738.15)


c:\Users\Andre Silva\Desktop\Dissertation25-26\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


    Saved plots to grid_search_plots/comparison_item26008_store6269_m1_lookback.png and grid_search_plots/comparison_item26008_store6269_m1_full.png

Processing Item 907969 Store 6269...
  Selected Seasonality Candidates (m): [1]
  >> Testing candidate m=1
  Running Lookback Approach...
Lookback Selected model: Order=(1, 0, 2), Seasonal=(0, 0, 0, 0), Trend=n (AIC=4434.46)


...

c:\Users\Andre Silva\Desktop\Dissertation25-26\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning:

Maximum Likelihood optimization failed to converge. Check mle_retvals



    Saved plots to grid_search_plots/comparison_item907969_store6269_m1_lookback.png and grid_search_plots/comparison_item907969_store6269_m1_full.png

Processing Item 907967 Store 6269...
  Selected Seasonality Candidates (m): [1]
  >> Testing candidate m=1
  Running Lookback Approach...
Lookback Selected model: Order=(1, 0, 1), Seasonal=(0, 0, 0, 0), Trend=n (AIC=3884.63)


c:\Users\Andre Silva\Desktop\Dissertation25-26\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning:

Maximum Likelihood optimization failed to converge. Check mle_retvals



  Running Full History Approach...
Full Selected model: Order=(1, 0, 1), Seasonal=(0, 0, 0, 0), Trend=n (AIC=5144.03)
    Saved plots to grid_search_plots/comparison_item907967_store6269_m1_lookback.png and grid_search_plots/comparison_item907967_store6269_m1_full.png
Calibration complete. Results saved to 'autoarima_comparison_results.csv'.


c:\Users\Andre Silva\Desktop\Dissertation25-26\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning:

Maximum Likelihood optimization failed to converge. Check mle_retvals

